# Stage 2 joint experiment notebook

Run order after Exp2 refactor: 00 prepare dataset → 01 Exp2A baseline → 02 Exp2B GCA baseline → 03 Exp2D lane detail neck → 04 Exp2E matched lane loss → 05 Exp2F detail + matching. Run 06 KD only if the teacher checkpoint exists. Run Exp3 notebooks only after Exp2F improves lane geometry. Every notebook re-extracts the Drive tar into `/content`; never assume files from a previous Colab runtime still exist. Training logs are printed in this notebook cell and mirrored to `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


# Stage 2 Notebook 03 - Exp2D RMT-GCA + lane detail neck

This is the first new Exp2 architecture test. It keeps the RMT-GCA detection path, but gives the lane branch a C2/stride-4 detail path fused into P3 before the CLRKD-style curve head. The goal is to reduce lane geometry error without hurting detection.

This notebook prints training output directly in the cell and mirrors the same text to Drive logs under `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)


Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys
CONFIG = 'stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)


[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp03_rmt_gca_lane_detail_joint_smoke.log
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys
CONFIG = 'stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'
cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--force-extract',
    '--print-every', '10',
]
print('About to run:', ' '.join(cmd), flush=True)
print('Expected output tar is defined inside', CONFIG, flush=True)
run_streaming(cmd, log_path=os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_train.log'))


About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --force-extract --print-every 10
Expected output tar is defined inside stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --force-extract --print-every 10
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp03_rmt_gca_lane_detail_joint_train.log
[run_streaming] return_code=0


0

## What to watch during training

- `val/lane_point_mae`: lower is better. This is the main geometry signal for the current Exp2 debugging.
- `val/lane_exist_acc`: checks whether lane existence is learned. High existence with flat MAE means geometry is still weak.
- `val/det_loss`: lower is better for vehicle detection.
- `train/mtl/lambda_lane_runtime` and `train/mtl/lambda_lane_epoch`: show the lane weight used in the joint loss.
- `gate/p3_lane_mean`, `gate/p4_lane_mean`, `gate/p5_lane_mean`: only in detail/GCA runs; these show how strongly the lane branch uses task-specific features per scale.
